In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/movie_data.csv', encoding='utf-8')

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.3,
    random_state=7
)

tfidf = TfidfVectorizer(dtype=np.float32)

X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)
from torch.utils.data import DataLoader, TensorDataset
import torch
from lib import *

# dataset = JointSparseDataset(X_train, y_train)
# data_loader = DataLoader(dataset, 64, shuffle=True, collate_fn= sparse_collate)

dataset = JointDataset(X_train, y_train)
data_loader = DataLoader(dataset, 64, shuffle=True)


In [2]:
from lib import NeuralNet

input_shape = X_train.shape[1]

model = NeuralNet(input_shape)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4, weight_decay= 0)
train_model(model, optimizer, data_loader)

# Evaluate the model

test_loader = DataLoader(JointDataset(X_test, y_test), batch_size=64, shuffle=False)
evaluate_model(model, data_loader, "Training")
evaluate_model(model, test_loader)


Device found: cuda
Train time = 34.79811356100254 sec
Training Accuracy: 0.9831142857142857
Test Accuracy: 0.9086666666666666


In [40]:
from lib import DynamicNeuralNet

input_shape = X_train.shape[1]

model_with_dropout = DynamicNeuralNet(input_shape, [64,64], [0.1, 0.5])
optimizer = torch.optim.Adam(model_with_dropout.parameters(), lr = 1e-4)
train_model(model_with_dropout, optimizer, data_loader)
# print(model_with_dropout)

# Evaluate the model
test_loader = DataLoader(JointDataset(X_test, y_test), batch_size=64, shuffle=False)
# evaluate_model(model_with_dropout, data_loader, "Training")
evaluate_model(model_with_dropout, test_loader)

Device found: cuda
Train time = 31.178214122002828 sec
Test Accuracy: 0.9082666666666667


In [12]:

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

def tokenizer(text):
    return text.split()

df = pd.read_csv('../data/movie_data.csv', encoding='utf-8')

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.3,
    random_state=7
)

tfidf = TfidfVectorizer(dtype=np.float32, ngram_range= (1,1), stop_words= None, tokenizer= tokenizer)
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)
lr = LogisticRegression(solver='liblinear', C = 10.0, penalty = 'l2' )
lr.fit(X_train, y_train)
y_p = lr.predict(X_test)


      

/home/alekh/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/alekh/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
